# Illustrated Fantasy Map to GeoTIFF

#### Description and Purpose

This is a simple tool for converting a .jpg image of a fantasy map to a georeferenced .tif file with a projected coordinate reference system and a scale of your choosing, overlaid on the surface of the Earth. This allows us to get around that fact that most fantasy worlds don't have robust geodesy information available. While it's possible that some fictional planets may be significantly larger or smaller than ours, most fantasy maps have features that are comparable in size to the continents on Earth. 

For now, the Web Mercator projection is used (EPSG:3857). This was chosen because it is familiar to many people and allows maps to be easily aligned with the European subcontinent, which many of them resemble anyway. Most crucially, we need to start with a "flat" projected coordinate system because our illustrated images are also flat. This avoids distortion due to the curvature of the Earth. Once the image has projected coordinates, it can be transformed into a geographic coordinate system later on. 

This tool could have many possible uses, but its main purpose is to provide an easy way for players of table top role playing games (TTRPGs) such as Dungeons & Dragons to calculate more precise distances and travel times and create custom map layers for their campaigns. Once converted and exported as a GeoTIFF, the user can subject the fantasy map to a wide range of geospatial operations in the GIS software of their choosing. 

#### How It Works
The scale bar of the fantasy map will be used to appropriately scale the image to the Earth's surface. If there is no scale bar, the user will need to invent a scale. Crucially, for the purposes of this tool, *the scale  is related to the resolution of the image*: when specifying the ratio of inches to miles, the inches represent real inches measured on the jpg when viewed at 100% zoom. It is therefore dependent on accurate knowledge of the image's dots per inch (DPI), which is read from the image's metadata using the Pillow module. However, in some cases, this metadata may be missing or inaccurate. In that case, the user may input a DPI manually, and trial and error may yield an appropriate relationship between DPI, image dimensions in inches, and scale in inches:miles. 

After specifying the file path of a .jpg image, a scale ratio, a DPI (if needed), and an output file path, the user can click the "Convert to GeoTIFF" button. For simplicity, this button initates the completion of all processing and output functions, so the user can input everything up front and then click only once. The image is converted to a .tif file, projected to Web Mercator, given a default center point, written to a file, and then displayed in Folium. 

### *For step by step instructions and to use the tool, please scroll to the end of this notebook.*

#### Inspiration
I built and tested this program using a map of the Sword Coast of Faerun, one of the main locations in the widely popular Forgotten Realms DnD setting. I was inspired to create this because of limitations I encountered when trying to georeference this map in ArcGIS Pro. Motivated by a Dungeon Master's complaints about calculating travel time, I found a high resolution JPG map of the region, converted it to a tif with a projected coordinate system, and attempted to line it up with Europe. However, I found that while I had options drag and resize the image or use ground control points, I couldn't easily find a simple means of setting the scale of the image in relation to its resolution or simply giving it a center point. This makes a lot of sense, since georeferencing typically involves aligning and rectifying real geographic features of Earth, although it's likely there's a way to do this that I simply missed in my googling. I eventually was able to scale the map correctly by repeatedly resizing it and then checking the length of the scale bar using the Measure tool. By the time I was finished, the DM told me he'd actually prefer to use a slightly different version of the map, so I decided I had no choice but to create a custom tool.  

#### Potential Future Features
This tool is essentially a Minimum Viable Product at this stage. With a bit more work, many other options and operations could be included. I'd like to make it possible to give the GeoTIFF a CRS and a center point of the user's choosing right from the start. This would make it so that different maps could be aligned with different Earth continents and multiple maps from the same fictional planet could be combined, overlaid together on different parts of the Earth's surface. Letting the user select between UTM zones could be useful, too, due to their small size and "flatness," characteristic they share with some illustrated fantasy maps. I initially tried to set this up, but switched to Web Mercator when issues with projection math caused intractable bugs for the Folium map display.

Given more time, I would look into ways to expand the DPI detection functionality and improve the user interface for allowing the user to choose whether it should be detected automatically or entered manually. It would also be nice to help the user determine the "real" length of the illustrated scale bar, if there is one - perhaps this could be done with a widget that displays the image at full resolution and overlays an inch-long measurement, allowing the user to click and drag the image to compare the scale bar to the inch. I am also interested in better input and output functions: could there be a folder where inputs can be saved and then selected from a dropdown menu? Better means of automatically formatting output names? A default output folder? Finally, 

## Setup

In [50]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from rasterio.transform import from_origin
import numpy as np
import rasterio
import folium
import rasterio.plot
import matplotlib.pyplot as plt
import tempfile
import base64
import io
from pyproj import Transformer

## Functions

### Image Processing

In [51]:
# Function for getting the dpi metadata from the image file itself if it's available
def get_image_dpi(filepath):
    # Open the file with the Image tool from the Pillow library 
    with Image.open(filepath) as img:
        # Try to get the dpi from the metadata
        dpi = img.info.get('dpi')
        if dpi:
            return dpi[0]  # DPI is usually a tuple (x_dpi, y_dpi), but scale bars are horizontal, so we only need the x DPI value
        else:
            return None  # DPI not embedded... This will trigger the display of an input widget 
        
# Image processing helper function to prepare an image to be saved as a geotiff
# gets called by the main save function when button is clicked  
def process_image(path, dpi, inches, miles):
    # Compute "real-world" scale of the map based on the scale bar
    scale_ratio = miles / inches  # miles per inch
    
    # open with Pillow
    with Image.open(path) as img:
        # Get the image width in pixels from the image size tuple
        width_px, height_px = img.size
        # Convert to a numpy array so rasterio can interact with it later on 
        img_array = np.array(img.convert("RGB"))  # Make sure it's in RGB format
    
    # Convert image width from pixels to inches using the DPI
    img_width_inches = width_px / dpi
    #Convert width from inches to miles using the scale ratio (based on user input)
    img_width_miles = img_width_inches * scale_ratio
    # convert from miles to meters because that's what our CRS uses
    img_width_meters = img_width_miles * 1609.34
    # Calculate pixel size in meters - That's Earth distance per pixel
    pixel_size_m = img_width_meters / width_px

    # We'll place the center at 0, 0 because the image isn't actually on Earth
    # These values work well for Web Mercator projection (EPSG:3857)
    center_lon = 0  # Longitude of prime meridian
    center_lat = 0  # Latitude of equator
    
    # Ok now we have to estimate the coordinates of the NW corner of the image
    # Converting degrees to meters, roughly... 111,320m/degree at equator
    west = center_lon - (img_width_meters / 2) / 111320
    north = center_lat + (height_px * pixel_size_m) / 2 / 111320
    
    # Affine transformation for georeferencing
    # Maps pixel coordinates to Earth meters in Web Mercator
    transform = from_origin(west * 111320, north * 111320, pixel_size_m, pixel_size_m)

    # Return a dictionary with the image data and georef info
    return {
        "array": img_array,
        "transform": transform,
        "crs": "EPSG:3857",
        "width": width_px,
        "height": height_px
    }

### Saving the GeoTIFF

In [52]:
# Function for saving the processed image as a georeferenced GeoTIFF
def save_geotiff(output_path):
    try:
        # Extract data from the dictionary prepared by the process_image function
        # georef_data will be set as a global variable by the save button click function
        array = georef_data['array'] # RGB image as a numpy array
        transform = georef_data['transform'] # Affine transformation
        crs = georef_data['crs'] # CRS. I could hardcode this value, but i want it to be flexible later
        
        # Open a new raster file
        with rasterio.open(
            output_path, # path
            'w', # write mode
            driver='GTiff', # file format
            height=array.shape[0], # height in pixels
            width=array.shape[1], # width in pixels
            count=3, # bands
            dtype=array.dtype, # data type??? hmm
            crs=crs, # Literally crs
            transform=transform # Affine again (would be a pretty name for a girl)
        ) as dst:
            # Writing each of the 3 bands. Rasterio is base 1
            for i in range(3):
                dst.write(array[:, :, i], i + 1)
        
        # Says whether the function was successfully and the name of the path
        return True, output_path

    except Exception as e:
        return False, str(e)

### Displaying the Map

In [53]:
# Function for displaying the output file as an interactive map overlay with Folium
def show_map():
    try:
        from pyproj import Transformer # module for CRS conversion
        
        # Let's get the georeferencing info
        transform = georef_data['transform'] # Affine transform to map pixels to meters
        width = georef_data['width'] # Image width in pixels
        height = georef_data['height'] # Image height in pixels
        crs = georef_data['crs'] # Current coordinate system (Web Mercator)
        array = georef_data['array'] # RGB image as a NumPy array

        # Calculate image bounds in original CRS (Web Mercator)
        top_left_x = transform.c # X (east-west) coordinate of the top-left corner
        top_left_y = transform.f # Y (north-south) coordinate of the top-left corner
        bottom_right_x = top_left_x + width * transform.a # Right edge
        bottom_right_y = top_left_y - height * transform.e # Bottom edge

        # Convert bounds from Web Mercator to EPSG:4326 (lat/lon)
        # Folium wants a geographic coordinate system for 
        transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
        west, north = transformer.transform(top_left_x, top_left_y)
        east, south = transformer.transform(bottom_right_x, bottom_right_y)

        # Format the bounds for Folium
        bounds = [[south, west], [north, east]]

        # Sanity check: make sure coordinate values are valid numbers and not infinite or missing
        if not all(np.isfinite(coord) for pair in bounds for coord in pair):
            raise ValueError(f"Non-finite values found in bounds: {bounds}")

        # Convert image to base64 JPEG string
        # Apparently we have to do this because the image isn't saved online?
        # Needs to be served from a server in order to create a folium/leaflet browser display
        # So instead we're embedding it directly into Folium's html
        # I didn't totally understand this part but I certainly tried... and it works. 
        img = Image.fromarray(array) # Convert from NumPy back to Pillow
        buffered = io.BytesIO() # Buffer in memory
        img.save(buffered, format="JPEG") # Save in JPEG format
        img_base64 = base64.b64encode(buffered.getvalue()).decode() # Encode to base64 which is what we need
        img_url = f"data:image/jpeg;base64,{img_base64}" # Build a data URL for embedding

        # Create and display interact map and image overlay
        m = folium.Map() # create blank map to get started
        folium.raster_layers.ImageOverlay(
            image=img_url, # Base64 string we just made
            bounds=bounds, # Bounds of image
            name="Fantasy Map", # Layer name
            opacity=0.7, # Make it a little transparent
            interactive=True, 
            cross_origin=False
        ).add_to(m)

        # automatically zoom map to fit image bounds
        m.fit_bounds(bounds)
        # Dunno if we need this cuz there's only one layer but I'm keeping it for flexibility
        folium.LayerControl().add_to(m)

        print("🗺️ Fantasy map overlay added to the globe:")
        display(m)

    except Exception as e:
        print(f"❌ Error displaying map with overlay: {e}")

## User Interface

### Gathering Parameters

In [54]:
# Some little instructions
intro_text = widgets.HTML("<b>🗺️ Fantasy Map Georeferencer</b><br>Enter a path to your map and scale information below.")
file_help = widgets.HTML("<i>Example: /home/user/maps/my_fantasy_map.jpg</i>")

# File path input widget
file_input = widgets.Text(
    value='',
    placeholder='Enter path to JPG file',
    description='File:',
    layout=widgets.Layout(width='60%')
)

# Title and description for scale input boxes
scale_text = widgets.HTML("<b>Scale:</b> Use the scale bar on your fantasy map or make up your own scale as a ratio of inches to miles.")

# Scale bar input
scale_inches = widgets.FloatText(
    value=1.0,
    description='Inches:',
    layout=widgets.Layout(width='50%')
)
scale_miles = widgets.FloatText(
    value=50.0,
    description='= Miles:',
    layout=widgets.Layout(width='50%')
)

# DPI input (hidden unless needed)
dpi_input = widgets.IntText(
    value=300,
    description='Horizontal DPI:',
    layout=widgets.Layout(width='50%')
)
dpi_box = widgets.VBox([dpi_input])
dpi_box.layout.display = 'none'  # hide initially

# Output display for messages
output = widgets.Output()

# Event handler when file path is updated
def on_file_change(change):
    with output:
        clear_output()
        # New file path that the user entered
        path = change['new']
        # tries to read DPI using the DPI function 
        dpi = get_image_dpi(path)
        if dpi:
            # acknowledges that it's just the horizontal value that we need
            print(f"✅ Detected horizontal DPI from image metadata: {dpi}")
            dpi_input.value = int(dpi)
            dpi_box.layout.display = 'none'
        else:
            # Shows the DPI input widget if the metadata is missing
            print("⚠️ Horizontal DPI not found in image metadata. Please enter it manually.")
            dpi_box.layout.display = 'block'

# Tells file_input widget to call this event handler whenever its value changes!!
file_input.observe(on_file_change, names='value')

# Text input for output filename
output_path_input = widgets.Text(
    # Default value
    value='fantasy_map_georef.tif',
    placeholder='Enter filename to save GeoTIFF',
    description='Output:',
    layout=widgets.Layout(width='70%')
)

### Button for Processing, Saving & Displaying

In [55]:
# Process, save & display button
save_button = widgets.Button(
    description='Convert to GeoTIFF',
    button_style='info',
    tooltip='Click to save the georeferenced image'
)

# Save button click handler
# This essential function calls the process_image, save_geotiff, and show_map functions
def on_save_button_click(button):
    # Set global variable to hold the processed image data
    global georef_data
    with output:
        clear_output()
        try:
            # Gather the data from the widgets and other functions
            path = file_input.value
            dpi = dpi_input.value
            inches = scale_inches.value
            miles = scale_miles.value

            # calls process_image function, passes it the values input by user in widgets
            # and saves it to the global variable 
            georef_data = process_image(path, dpi, inches, miles)
            print("✅ Image and georeferencing info ready!")

            # sets output path based on user input widget
            output_path = output_path_input.value
            # If process_image worked, the save function is called 
            success, result = save_geotiff(output_path)

            if success:
                print(f"✅ File saved successfully to: {result}")
                # Warn the user it takes a bit
                print(f"⏳ Please wait while map preview is loaded.")
                # Call the folium map display function
                show_map()
            else:
                print(f"❌ Failed to save file: {result}")

        except Exception as e:
            print(f"❌ Error: {e}")

save_button.on_click(on_save_button_click)

In [56]:
# Layout using VBox
# This is a really nifty trick I wish I knew about earlier!!!
# All the main function needs to do is display this layout, and the rest just happens
input_ui = widgets.VBox([
    intro_text,
    file_help,
    file_input,
    scale_text,
    widgets.HBox([scale_inches, scale_miles]),
    dpi_box,
    output_path_input,
    save_button,    
    output
])

## Instructions

1. Save a .jpg image of an illustrated fantasy map to a known path. If you save it to the same directory as the notebook, all you need to input is the file name. 
2. If prompted, enter the image's DPI. If you're not sure, try 72 or 96.
3. Expand your image to full resolution in an image viewer and measure or estimate the length of the scale bar. Use this to enter the Inches:Miles scale ratio. If there is no scale bar, invent a scale. If you're not sure, try 1:50 or 1:100. For more info, see the "How It Works" section at the beginning of this notebook.
4. Specify an output file name. 
5. Click "Convert to GeoTIFF" and enjoy! 

## Execution

In [57]:
display(input_ui)